# 🎬 YouTube Chatbot using LangChain | RAG System

**Steps:**
1. Fetch the YouTube transcript (or paste it manually)
2. Split the text into chunks
3. Create embeddings and store them in FAISS
4. Retrieve relevant chunks based on the question
5. Generate the answer using GPT-4.1-mini

## Step 0 — Libraries Install Karo

In [1]:
!pip install -q youtube-transcript-api==0.6.3 langchain-community langchain-openai faiss-cpu tiktoken python-dotenv
print("Done!")

Done!


## Step 0b — Enter OpenAI API

In [2]:
import os
from getpass import getpass
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # OMP conflict fix

print("OMP fix applied!")

OMP fix applied!


In [3]:
from dotenv import load_dotenv
load_dotenv()  # .env file se key automatically load hogi
print("API Key loaded!")

API Key loaded!


## Step 0c — Imports

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
print("All imports successful!")

C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


All imports successful!


## Step 1a — Get the Transcript

### Method A — Automatic (try this first)
If your Colab IP is not blocked, this method should work.

### Method B — Manual Paste (if Method A fails)
1. Open the YouTube video  
2. Click on the `...` (three dots) below the video  
3. Click on **"Open Transcript"**  
4. Copy the entire transcript  
5. Paste it into the cell below  

In [ ]:
# ─── METHOD A : Automatic fetch ───────────────────────────────────────────────
# If this method fails (error) then use Method B below (manual paste)

video_id = "Gfr50f6ZBvo"   # only id of your selected video, not the full URL
transcript = ""

try:
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(f"SUCCESS! Transcript fetch!")
    print(f"Total characters: {len(transcript)}")
    print("\nPehle 300 characters:")
    print(transcript[:300])

except TranscriptsDisabled:
    print("Transcripts disabled. Use Method B.")

except Exception as e:
    print(f"Method A failed: {type(e).__name__}")
    print("=> Run Method B cell given below")

Method A fail hua: ParseError
=> Neeche Method B wali cell chalao


In [46]:
# ─── METHOD B : Manual Paste ───────────────────────────────────────────────────
# Run only when method A fails.
# YouTube se transcript copy karke neeche teen quotes ke beech paste karo
# Download the transcript from YouTube, copy it, and paste it here between the triple quotes.

transcript = """
Imagine you happen across a short movie script that
describes a scene between a person and their AI assistant.
The script has what the person asks the AI, but the AI's response has been torn off.
Suppose you also have this powerful magical machine that can take
any text and provide a sensible prediction of what word comes next.
You could then finish the script by feeding in what you have to the machine,
seeing what it would predict to start the AI's answer,
and then repeating this over and over with a growing script completing the dialogue.
When you interact with a chatbot, this is exactly what's happening.
A large language model is a sophisticated mathematical function
that predicts what word comes next for any piece of text.
Instead of predicting one word with certainty, though,
what it does is assign a probability to all possible next words.
To build a chatbot, you lay out some text that describes an interaction between a user
and a hypothetical AI assistant, add on whatever the user types in as the first part of
the interaction, and then have the model repeatedly predict the next word that such a
hypothetical AI assistant would say in response, and that's what's presented to the user.
In doing this, the output tends to look a lot more natural if
you allow it to select less likely words along the way at random.
So what this means is even though the model itself is deterministic,
a given prompt typically gives a different answer each time it's run.
Models learn how to make these predictions by processing an enormous amount of text,
typically pulled from the internet.
For a standard human to read the amount of text that was used to train GPT-3,
for example, if they read non-stop 24-7, it would take over 2600 years.
Larger models since then train on much, much more.
You can think of training a little bit like tuning the dials on a big machine.
The way that a language model behaves is entirely determined by these
many different continuous values, usually called parameters or weights.
Changing those parameters will change the probabilities
that the model gives for the next word on a given input.
What puts the large in large language model is how
they can have hundreds of billions of these parameters.
No human ever deliberately sets those parameters.
Instead, they begin at random, meaning the model just outputs gibberish,
but they're repeatedly refined based on many example pieces of text.
One of these training examples could be just a handful of words,
or it could be thousands, but in either case, the way this works is to
pass in all but the last word from that example into the model and
compare the prediction that it makes with the true last word from the example.
An algorithm called backpropagation is used to tweak all of the parameters
in such a way that it makes the model a little more likely to choose
the true last word and a little less likely to choose all the others.
When you do this for many, many trillions of examples,
not only does the model start to give more accurate predictions on the training data,
but it also starts to make more reasonable predictions on text that it's never
seen before.
Given the huge number of parameters and the enormous amount of training data,
the scale of computation involved in training a large language model is mind-boggling.
To illustrate, imagine that you could perform one
billion additions and multiplications every single second.
How long do you think it would take for you to do all of the
operations involved in training the largest language models?
Do you think it would take a year?
Maybe something like 10,000 years?
The answer is actually much more than that.
It's well over 100 million years.
This is only part of the story, though.
This whole process is called pre-training.
The goal of auto-completing a random passage of text from the
internet is very different from the goal of being a good AI assistant.
To address this, chatbots undergo another type of training,
just as important, called reinforcement learning with human feedback.
Workers flag unhelpful or problematic predictions,
and their corrections further change the model's parameters,
making them more likely to give predictions that users prefer.
Looking back at the pre-training, though, this staggering amount of
computation is only made possible by using special computer chips that
are optimized for running many operations in parallel, known as GPUs.
However, not all language models can be easily parallelized.
Prior to 2017, most language models would process text one word at a time,
but then a team of researchers at Google introduced a new model known as the transformer.
Transformers don't read text from the start to the finish,
they soak it all in at once, in parallel.
The very first step inside a transformer, and most other language models for that matter,
is to associate each word with a long list of numbers.
The reason for this is that the training process only works with continuous values,
so you have to somehow encode language using numbers,
and each of these lists of numbers may somehow encode the meaning of the
corresponding word.
What makes transformers unique is their reliance
on a special operation known as attention.
This operation gives all of these lists of numbers a chance to talk to one another
and refine the meanings they encode based on the context around, all done in parallel.
For example, the numbers encoding the word bank might be changed based on the
context surrounding it to somehow encode the more specific notion of a riverbank.
Transformers typically also include a second type of operation known
as a feed-forward neural network, and this gives the model extra
capacity to store more patterns about language learned during training.
All of this data repeatedly flows through many different iterations of
these two fundamental operations, and as it does so,
the hope is that each list of numbers is enriched to encode whatever
information might be needed to make an accurate prediction of what word
follows in the passage.
At the end, one final function is performed on the last vector in this sequence,
which now has had a chance to be influenced by all the other context from the input text,
as well as everything the model learned during training,
to produce a prediction of the next word.
Again, the model's prediction looks like a probability for every possible next word.
Although researchers design the framework for how each of these steps work,
it's important to understand that the specific behavior is an emergent phenomenon
based on how those hundreds of billions of parameters are tuned during training.
This makes it incredibly challenging to determine
why the model makes the exact predictions that it does.
What you can see is that when you use large language model predictions to autocomplete
a prompt, the words that it generates are uncannily fluent, fascinating, and even useful.
If you're a new viewer and you're curious about more details on how
transformers and attention work, boy do I have some material for you.
One option is to jump into a series I made about deep learning,
where we visualize and motivate the details of attention and all the other steps
in a transformer.
Also, on my second channel I just posted a talk I gave a couple
months ago about this topic for the company TNG in Munich.
Sometimes I actually prefer the content I make as a casual talk rather than a produced
video, but I leave it up to you which one of these feels like the better follow-on.
"""

transcript = transcript.strip()
print(f"Transcript ready! Total characters: {len(transcript)}")
print("\nFirst 300 characters:")
print(transcript[:300])

Transcript ready! Total characters: 7524

First 300 characters:
Imagine you happen across a short movie script that
describes a scene between a person and their AI assistant.
The script has what the person asks the AI, but the AI's response has been torn off.
Suppose you also have this powerful magical machine that can take
any text and provide a sensible predic


## Step 1b — Text Splitting


Break down the complete transcript into small chunks so that our LLM can process all.

In [15]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

print(f"Total chunks: {len(chunks)}")

Total chunks: 10


In [44]:
# chekc one simple chunk
chunks[0]

Document(metadata={}, page_content="Imagine you happen across a short movie script that\ndescribes a scene between a person and their AI assistant.\nThe script has what the person asks the AI, but the AI's response has been torn off.\nSuppose you also have this powerful magical machine that can take\nany text and provide a sensible prediction of what word comes next.\nYou could then finish the script by feeding in what you have to the machine,\nseeing what it would predict to start the AI's answer,\nand then repeating this over and over with a growing script completing the dialogue.\nWhen you interact with a chatbot, this is exactly what's happening.\nA large language model is a sophisticated mathematical function\nthat predicts what word comes next for any piece of text.\nInstead of predicting one word with certainty, though,\nwhat it does is assign a probability to all possible next words.\nTo build a chatbot, you lay out some text that describes an interaction between a user")

## Step 1c & 1d — Embeddings + FAISS Vector Store

convert each chunk into number(vector) and store into FAISS database

In [21]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

In [23]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector store ready! Total vectors: {len(vector_store.index_to_docstore_id)}")

Vector store ready! Total vectors: 10


In [42]:
#check the index mapping of Vector store
vector_store.index_to_docstore_id

{0: '85254321-f465-491b-8886-bc7fe8582747',
 1: 'a480ecf2-0ac4-4ad2-835e-b7cd37c47f96',
 2: 'd7509054-122c-4792-824d-9ab810b29876',
 3: 'da68ea12-4b24-48e8-a161-5e99cff086e3',
 4: 'f1be84e4-6433-4aa5-a79d-5cf0e95ca6d1',
 5: '967d21a7-e81a-4e38-9725-abc09db3bd1f',
 6: '210f4d3c-a511-449d-8643-5fed750bdade',
 7: 'fddac21b-20c8-4181-a8bd-979d0eb05d6a',
 8: 'fa46acb1-8996-4745-8ef0-8072172becac',
 9: '7cdf5aa7-1591-4681-9309-6e01aa1073a0'}

## Step 2 — Retrieval

creating the reteriever to find the relevent chunks from the vector store 

In [25]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [26]:
# Retriever test karo
retriever.invoke('What is deepmind')

[Document(id='fa46acb1-8996-4745-8ef0-8072172becac', metadata={}, page_content="Again, the model's prediction looks like a probability for every possible next word.\nAlthough researchers design the framework for how each of these steps work,\nit's important to understand that the specific behavior is an emergent phenomenon\nbased on how those hundreds of billions of parameters are tuned during training.\nThis makes it incredibly challenging to determine\nwhy the model makes the exact predictions that it does.\nWhat you can see is that when you use large language model predictions to autocomplete\na prompt, the words that it generates are uncannily fluent, fascinating, and even useful.\nIf you're a new viewer and you're curious about more details on how\ntransformers and attention work, boy do I have some material for you.\nOne option is to jump into a series I made about deep learning,\nwhere we visualize and motivate the details of attention and all the other steps\nin a transformer.\

## Step 3 — Augmentation

LLM and Prompt template setup.

In [27]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [28]:
prompt = PromptTemplate(
    template="""You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables=['context', 'question']
)

In [29]:
question       = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [30]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"where we visualize and motivate the details of attention and all the other steps\nin a transformer.\nAlso, on my second channel I just posted a talk I gave a couple\nmonths ago about this topic for the company TNG in Munich.\nSometimes I actually prefer the content I make as a casual talk rather than a produced\nvideo, but I leave it up to you which one of these feels like the better follow-on.\n\nAgain, the model's prediction looks like a probability for every possible next word.\nAlthough researchers design the framework for how each of these steps work,\nit's important to understand that the specific behavior is an emergent phenomenon\nbased on how those hundreds of billions of parameters are tuned during training.\nThis makes it incredibly challenging to determine\nwhy the model makes the exact predictions that it does.\nWhat you can see is that when you use large language model predictions to autocomplete\na prompt, the words that it generates are uncannily fluent, fascinating, a

In [31]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [32]:
final_prompt

StringPromptValue(text="You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      where we visualize and motivate the details of attention and all the other steps\nin a transformer.\nAlso, on my second channel I just posted a talk I gave a couple\nmonths ago about this topic for the company TNG in Munich.\nSometimes I actually prefer the content I make as a casual talk rather than a produced\nvideo, but I leave it up to you which one of these feels like the better follow-on.\n\nAgain, the model's prediction looks like a probability for every possible next word.\nAlthough researchers design the framework for how each of these steps work,\nit's important to understand that the specific behavior is an emergent phenomenon\nbased on how those hundreds of billions of parameters are tuned during training.\nThis makes it incredibly challenging to determine\nwhy the model makes the exact predict

## Step 4 — Generation

In [33]:
answer = llm.invoke(final_prompt)
print(answer.content)

I don't know.


## Chain creation all toghter

In [34]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [35]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [36]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [37]:
parallel_chain.invoke('who is Demis')

{'context': "where we visualize and motivate the details of attention and all the other steps\nin a transformer.\nAlso, on my second channel I just posted a talk I gave a couple\nmonths ago about this topic for the company TNG in Munich.\nSometimes I actually prefer the content I make as a casual talk rather than a produced\nvideo, but I leave it up to you which one of these feels like the better follow-on.\n\nAgain, the model's prediction looks like a probability for every possible next word.\nAlthough researchers design the framework for how each of these steps work,\nit's important to understand that the specific behavior is an emergent phenomenon\nbased on how those hundreds of billions of parameters are tuned during training.\nThis makes it incredibly challenging to determine\nwhy the model makes the exact predictions that it does.\nWhat you can see is that when you use large language model predictions to autocomplete\na prompt, the words that it generates are uncannily fluent, fa

In [38]:
parser = StrOutputParser()

In [39]:
main_chain = parallel_chain | prompt | llm | parser

In [40]:
main_chain.invoke('Can you summarize the video')

'The video discusses the workings of transformers and attention mechanisms in deep learning, emphasizing how large language models predict the next word in a sequence by assigning probabilities to all possible options. It highlights the emergent behavior of these models based on their training and the challenges in understanding their predictions. The speaker mentions a series on deep learning that visualizes these concepts and also references a casual talk posted on their second channel. Additionally, the video explains how transformers encode language using numerical representations and how attention allows these representations to interact and refine their meanings based on context.'

## Chatbot Loop — ASK ANY QUESTION

In [47]:
print("YouTube RAG Chatbot Ready!")
print("Apna question likho. Bahar jaane ke liye 'quit' likho.")
print("-" * 60)

while True:
    user_question = input("You: ").strip()

    if not user_question:
        continue

    if user_question.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break

    try:
        answer = main_chain.invoke(user_question)
        print(f"\nBot: {answer}\n")
        print("-" * 60)
    except Exception as e:
        print(f"Error: {type(e).__name__}: {e}")

YouTube RAG Chatbot Ready!
Apna question likho. Bahar jaane ke liye 'quit' likho.
------------------------------------------------------------

Bot: A large language model (LLM) is a sophisticated mathematical function that predicts what word comes next for any piece of text. It assigns a probability to all possible next words based on the input it receives.

------------------------------------------------------------

Bot: I don't know.

------------------------------------------------------------

Bot: Transformers are a type of language model that use a special operation known as attention to associate each word with a long list of numbers, encoding the meaning of the words based on their context. They also include a feed-forward neural network to enhance the model's capacity to store patterns about language learned during training. The model processes data through iterations of these operations to predict the next word in a passage, producing probabilities for every possible next 